# Accident detection

## Libraries & Dataset

In [1]:
!pip install -qU roboflow ultralytics wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 109.3 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("RoboFlow")
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-hossam (ahmed-hossam-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("accident-detection-ffdrf").project("accident-detection-8dvh5")
version = project.version(2)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Accident-Detection-2 in yolov8:: 100%|██████████| 32282/32282 [00:04<00:00, 7447.64it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# ── W&B: Initialize run with full hyperparameter config ──────────────
EPOCHS = 60
IMGSZ  = 640
BATCH  = 16
MODEL  = "yolov8n.pt"
PROJECT = "Accident_Severity_Detection"
RUN_NAME = "v1_Accident_detection_AG"     # edit AG to your short name & start with v1 & edit the version number as you go !!!!

run = wandb.init(
    project=PROJECT,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL,
        "pretrained":   True,
        # Training
        "epochs":          EPOCHS,
        "imgsz":           IMGSZ,
        "batch":           BATCH,
        "fraction":        1.0,     # Here we used 100% of the data
        
        # Dataset
        "dataset":         "accident-detection-8dvh5",
        "dataset_version": 2,
        "dataset_link":    "https://universe.roboflow.com/accident-detection-ffdrf/accident-detection-8dvh5/dataset/2",
        "num_classes":     2,
}
)
print(f"W&B run started: {run.url}")


W&B run started: https://wandb.ai/ahmed-hossam-suez-canal-university/Accident_Severity_Detection/runs/cksumif9


## Modeling

In [6]:
from ultralytics import YOLO

cfg = wandb.config  # use values logged to W&B

model = YOLO(cfg.model)

results = model.train(
    data='/kaggle/working/Accident-Detection-2/data.yaml',
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    fraction=cfg.fraction,
    
    project=PROJECT,
    name=RUN_NAME,
    plots=True,
)

Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Accident-Detection-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v1_Accident_detection_AG-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

In [7]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/box_loss":     metrics_dict.get("val/box_loss",         0),
    "final/cls_loss":     metrics_dict.get("val/cls_loss",         0),
    "final/dfl_loss":     metrics_dict.get("val/dfl_loss",         0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


Logged metrics:
  final/precision: 0.9788
  final/recall: 0.9735
  final/mAP50: 0.9814
  final/mAP50-95: 0.9121
  final/box_loss: 0.0000
  final/cls_loss: 0.0000
  final/dfl_loss: 0.0000
  final/fitness: 0.9121


In [8]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


  ✓ confusion_matrix
  ✓ confusion_matrix_normalized
  ✓ BoxPR_curve
  ✓ BoxF1_curve
  ✓ BoxP_curve
  ✓ BoxR_curve
  ✓ results
  ✓ labels
  ✓ val_batch0_labels
  ✓ val_batch0_pred
  ✓ val_batch1_labels
  ✓ val_batch1_pred
  ✓ val_batch2_labels
  ✓ val_batch2_pred

Logged 14 images/plots to W&B.


In [9]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name="Accident_detector",
    type="model",
    description="YOLOv8n fine-tuned for Accidents",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
        "dataset_link": cfg.dataset_link,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


Model artifact logged: license_plate_detector:v0:v0


In [10]:
# Finish the WandB run
wandb.finish()

final/box_loss,▁
final/cls_loss,▁
final/dfl_loss,▁
final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/box_loss,0
final/cls_loss,0
final/dfl_loss,0
